In [2]:
import os,sys,time
from glob import glob
#import cmocean as cmo
import xarray as xr
#from xgcm import Grid
import xesmf
import numpy as np
import matplotlib.pyplot as plt
from mom6_OBC_routines import *
from datetime import datetime
import netCDF4
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

### Inputs

In [3]:
#--- year to generate obc for:
year = '2005'
#--- output dir for unformatted OBC data:
out_obc_no_format_dir = year+'_raw/'
#--- output dir for formatted OBC data:
out_obc_format_dir = year+'/'
#--- check if outdirs exist/create if not:
make_dir(out_obc_no_format_dir)
make_dir(out_obc_format_dir)

#---mom6 horizontal grid:
target_horizontal_grid = xr.open_dataset('/glade/campaign/cgd/oce/projects/CROCODILE/MOM6/grid/NWA12/ocean_hgrid.nc').rename({'x': 'lon', 'y': 'lat'})

#--- mom6 grid topography file:
topo = xr.open_dataset('/glade/campaign/cgd/oce/projects/CROCODILE/MOM6/grid/NWA12/ocean_topo.nc')

#--- path to data to generate OBCs:
ds_in_path = '/glade/campaign/cgd/oce/projects/CROCODILE/glorys012/GLOBAL/'+year+'/'
ds_in_files = sorted(glob(os.path.join(ds_in_path,f'mercator*.nc')))

#--- var names:
#--- names of variable on source data:
var_names = ['so','thetao','uo','vo','zos']
#for renaming after all interpolations are done:
new_var_names = ['salt','temp','u','v','zeta'] 

#---OBC segments:
segs = [1,2,4]
segs_names = ['001','002','004']

### Extract boundary conditions:

In [4]:
drop_var_lst = ['mlotst','bottomT','sithick','siconc','usi','vsi']

In [5]:
lon_min = np.floor(target_horizontal_grid.lon.min()).values - 1
lon_max = np.ceil(target_horizontal_grid.lon.max()).values + 1

In [6]:
lat_min = np.floor(target_horizontal_grid.lat.min()).values - 1
lat_max = np.ceil(target_horizontal_grid.lat.max()).values + 1

In [ ]:
for fl in range(len(ds_in_files)):
    print(ds_in_files[fl])    
    ds_in = xr.open_dataset(ds_in_files[fl],decode_times=False).drop_vars(drop_var_lst).sel(latitude=slice(lat_min,lat_max),longitude=slice(lon_min,lon_max))

    #regrid data:
    regridded = regrid_tracer_obc(ds_in,target_horizontal_grid)

    if fl==0:
        ## depth to thickness:
        zl=ds_in.depth
        zi=0.5*(np.roll(zl,shift=-1)+zl)
        zi[-1]=6500.
        inter=zi
        dz=zi-np.roll(zi,shift=1)
        dz[0]=zi[0]
        inter=dz
        ## segment indices:
        nx, ny = tuple(regridded.dims[d] for d in ['nxp', 'nyp'])
        nz = ds_in.dims['depth']
        ny_segment_001 = np.array(range(ny))
        nx_segment_001 = np.array(range(1))
        ny_segment_002 = np.array(range(1))
        nx_segment_002 = np.array(range(nx))
        ny_segment_004 = np.array(range(1))
        nx_segment_004 = np.array(range(nx))
        nz_segment_001 = np.array(range(nz))
        nz_segment_002 = nz_segment_001.copy()
        nz_segment_004 = nz_segment_001.copy()
        #lat/lon arrays:
        # lon_segment_001 = regridded['lon'][:,-1].values
        # lat_segment_001 = regridded['lat'][:,-1].values
        # lon_segment_002 = regridded['lon'][-1,:].values
        # lat_segment_002 = regridded['lat'][-1,:].values
        # lon_segment_004 = regridded['lon'][0,:].values
        # lat_segment_004 = regridded['lat'][0,:].values
    
    #loop over each variable for each segment/boundary:
    for var in range(len(var_names)):
        for s in range(len(segs)):
            #extract boundary and fill land:
            if var_names[var]=='zos':
                vars()[new_var_names[var]+'_00'+str(segs[s])] = get_segment_fill_zeta(segs[s],regridded[var_names[var]])
            else:
                vars()[new_var_names[var]+'_00'+str(segs[s])] = get_segment_fill(segs[s],regridded[var_names[var]])

            #concatenate:
            if fl == 0:
                vars()['combined_'+new_var_names[var]+'_00'+str(segs[s])] = vars()[new_var_names[var]+'_00'+str(segs[s])]
            else:
                vars()['combined_'+new_var_names[var]+'_00'+str(segs[s])] = xr.concat([vars()['combined_'+new_var_names[var]+'_00'+str(segs[s])],vars()[new_var_names[var]+'_00'+str(segs[s])]],dim='time')

/glade/campaign/cgd/oce/projects/CROCODILE/glorys012/GLOBAL/2005/mercatorglorys12v1_gl12_mean_20050101_R20050105.nc


/glade/derecho/scratch/manishrv/tmp/ipykernel_127445/2196953971.py:3: DeprecationWarning: dropping variables using `drop` is deprecated; use drop_vars.
  ds_in = xr.open_dataset(ds_in_files[fl],decode_times=False).drop(drop_var_lst).sel(latitude=slice(lat_min,lat_max),longitude=slice(lon_min,lon_max))


/glade/campaign/cgd/oce/projects/CROCODILE/glorys012/GLOBAL/2005/mercatorglorys12v1_gl12_mean_20050102_R20050105.nc


/glade/derecho/scratch/manishrv/tmp/ipykernel_127445/2196953971.py:3: DeprecationWarning: dropping variables using `drop` is deprecated; use drop_vars.
  ds_in = xr.open_dataset(ds_in_files[fl],decode_times=False).drop(drop_var_lst).sel(latitude=slice(lat_min,lat_max),longitude=slice(lon_min,lon_max))


/glade/campaign/cgd/oce/projects/CROCODILE/glorys012/GLOBAL/2005/mercatorglorys12v1_gl12_mean_20050103_R20050105.nc


...

/glade/derecho/scratch/manishrv/tmp/ipykernel_127445/2196953971.py:3: DeprecationWarning: dropping variables using `drop` is deprecated; use drop_vars.
  ds_in = xr.open_dataset(ds_in_files[fl],decode_times=False).drop(drop_var_lst).sel(latitude=slice(lat_min,lat_max),longitude=slice(lon_min,lon_max))


/glade/campaign/cgd/oce/projects/CROCODILE/glorys012/GLOBAL/2005/mercatorglorys12v1_gl12_mean_20050307_R20050309.nc


/glade/derecho/scratch/manishrv/tmp/ipykernel_127445/2196953971.py:3: DeprecationWarning: dropping variables using `drop` is deprecated; use drop_vars.
  ds_in = xr.open_dataset(ds_in_files[fl],decode_times=False).drop(drop_var_lst).sel(latitude=slice(lat_min,lat_max),longitude=slice(lon_min,lon_max))


/glade/campaign/cgd/oce/projects/CROCODILE/glorys012/GLOBAL/2005/mercatorglorys12v1_gl12_mean_20050308_R20050309.nc


/glade/derecho/scratch/manishrv/tmp/ipykernel_127445/2196953971.py:3: DeprecationWarning: dropping variables using `drop` is deprecated; use drop_vars.
  ds_in = xr.open_dataset(ds_in_files[fl],decode_times=False).drop(drop_var_lst).sel(latitude=slice(lat_min,lat_max),longitude=slice(lon_min,lon_max))


In [11]:
print(regridded)

<xarray.Dataset> Size: 4GB
Dimensions:  (time: 1, nyp: 1561, nxp: 1481, depth: 50)
Coordinates:
  * depth    (depth) float32 200B 0.494 1.541 2.646 ... 5.275e+03 5.728e+03
  * time     (time) float32 4B 4.821e+05
Dimensions without coordinates: nyp, nxp
Data variables:
    zos      (time, nyp, nxp) float64 18MB 0.2806 0.2806 ... -0.9228 -0.9216
    thetao   (time, depth, nyp, nxp) float64 925MB 26.55 26.56 26.57 ... nan nan
    so       (time, depth, nyp, nxp) float64 925MB 33.82 33.81 33.8 ... nan nan
    uo       (time, depth, nyp, nxp) float64 925MB 0.05053 0.04558 ... nan nan
    vo       (time, depth, nyp, nxp) float64 925MB 0.13 0.1198 ... nan nan
Attributes:
    regrid_method:  bilinear


In [ ]:
### save unformatted data:
for var in range(len(var_names)):   
    for s in range(len(segs)):
        vars()['combined_'+new_var_names[var]+'_00'+str(segs[s])].to_netcdf(out_obc_no_format_dir + 'combined_'+new_var_names[var]+'_00'+str(segs[s])+'_'+year+'.nc')

In [ ]:
#format and save:
for s in range(len(segs)):
    for var in range(len(var_names)):
        if segs[s] == 1:
            if new_var_names[var] != 'zeta':
                da =  vars()['combined_'+new_var_names[var]+'_00'+str(segs[s])].rename({'depth':'nz_segment_001','nyp':'ny_segment_001'}).expand_dims(dim={'nx_segment_001':nx_segment_001})
                inter_z = inter.rename({'depth':'nz_segment_001'})
                inter_z = inter_z.expand_dims({'nx_segment_001':nx_segment_001,'time':da.time.values,'ny_segment_001':ny_segment_001})
                inter_z = inter_z.transpose('nx_segment_001','time','nz_segment_001','ny_segment_001')
                ds = xr.Dataset(
                    data_vars = dict(
                        {new_var_names[var]+'_segment_001' : (['nx_segment_001','time','nz_segment_001','ny_segment_001'],da.data),
                        'dz_'+new_var_names[var]+'_segment_001' : (['nx_segment_001','time','nz_segment_001','ny_segment_001'],inter_z.data),
                        'lon_segment_001' : (['lon_segment_001'],lon_segment_001),
                        'lat_segment_001' : (['lat_segment_001'],lat_segment_001),
                        }),
                    coords = dict(
                        time = (['time'],da.time.data),
                        nz_segment_001 = (['nz_segment_001'], nz_segment_001),
                        ny_segment_001 = (['ny_segment_001'], ny_segment_001),
                        nx_segment_001 = (['nx_segment_001'], nx_segment_001),))
                ds = ds.transpose('time','nz_segment_001','ny_segment_001','nx_segment_001','lon_segment_001','lat_segment_001')
                ds.time.attrs['calendar']='gregorian'
                ds.time.attrs['units']='hours since 1950-01-01'
                ds.time.encoding['_FillValue']=1.e20 
                ds.lon_segment_001.encoding['_FillValue']=1.e20
                ds.lat_segment_001.encoding['_FillValue']=1.e20
                ds.ny_segment_001.encoding['_FillValue']=-10000
                ds.nz_segment_001.encoding['_FillValue']=-10000
                ds.nx_segment_001.encoding['_FillValue']=-10000
                for v in ds:
                        ds[v].encoding['_FillValue']=1.e20 
                ds.to_netcdf(out_obc_format_dir+'combined_'+new_var_names[var]+'_00'+str(segs[s])+'_'+year+'.nc',unlimited_dims=('time'))
            else:
                nz_zeta =np.linspace(0,1,1)
                dz_zeta = inter_z[:,:,0:1:,:].transpose('nx_segment_001','nz_segment_001','time','ny_segment_001')
                da =  vars()['combined_'+new_var_names[var]+'_00'+str(segs[s])].rename({'nyp':'ny_segment_001'}).expand_dims(dim={'nz_segment_001':nz_zeta,'nx_segment_001':nx_segment_001})
                ds = xr.Dataset(
                        data_vars = dict(
                            zeta_segment_001 = (['nx_segment_001','nz_segment_001','time','ny_segment_001'],da.data),
                            dz_zeta_segment_001 = (['nx_segment_001','nz_segment_001','time','ny_segment_001'],dz_zeta.data),
                            lon_segment_001 = (['lon_segment_001'],lon_segment_001),
                            lat_segment_001 = (['lat_segment_001'],lat_segment_001),
                        ),
                        coords = dict(
                            time = (['time'],da.time.data),
                            nz_segment_001 = (['nz_segment_001'], nz_zeta),
                            ny_segment_001 = (['ny_segment_001'], ny_segment_001),
                            nx_segment_001 = (['nx_segment_001'], nx_segment_001),))
                ds.time.attrs['calendar']='gregorian'
                ds.time.attrs['units']='days since 1999-12-31'
                ds.time.encoding['_FillValue']=1.e20 
                ds.lon_segment_001.encoding['_FillValue']=1.e20
                ds.lat_segment_001.encoding['_FillValue']=1.e20
                ds.ny_segment_001.encoding['_FillValue']=-10000
                ds.nz_segment_001.encoding['_FillValue']=-10000
                ds.nx_segment_001.encoding['_FillValue']=-10000
                for v in ds:
                        ds[v].encoding['_FillValue']=1.e20  
                ds = ds.transpose('time','nz_segment_001','ny_segment_001','nx_segment_001','lon_segment_001','lat_segment_001')
                ds.to_netcdf(out_obc_format_dir+'combined_'+new_var_names[var]+'_00'+str(segs[s])+'_'+year+'.nc',unlimited_dims=('time'))
        elif segs[s] == 2:
            if new_var_names[var] != 'zeta':
                da =  vars()['combined_'+new_var_names[var]+'_00'+str(segs[s])].rename({'depth':'nz_segment_002','nxp':'nx_segment_002'}).expand_dims(dim={'ny_segment_002':ny_segment_002})
                inter_z = inter.rename({'depth':'nz_segment_002'})
                inter_z = inter_z.expand_dims({'ny_segment_002':ny_segment_002,'time':da.time.values,'nx_segment_002':nx_segment_002})
                inter_z = inter_z.transpose('ny_segment_002','time','nz_segment_002','nx_segment_002')
                ds = xr.Dataset(
                    data_vars = dict(
                        {new_var_names[var]+'_segment_002' : (['ny_segment_002','time','nz_segment_002','nx_segment_002'],da.data),
                        'dz_'+new_var_names[var]+'_segment_002': (['ny_segment_002','time','nz_segment_002','nx_segment_002'],inter_z.data),
                        'lon_segment_002' : (['lon_segment_002'],lon_segment_002),
                        'lat_segment_002' : (['lat_segment_002'],lat_segment_002),
                        }),
                    coords = dict(
                        time = (['time'],da.time.values),
                        nz_segment_002 = (['nz_segment_002'], nz_segment_002),
                        ny_segment_002 = (['ny_segment_002'], ny_segment_002),
                        nx_segment_002 = (['nx_segment_002'], nx_segment_002),))
                ds.time.attrs['calendar']='gregorian'
                ds.time.attrs['units']='days since 1999-12-31'
                ds.time.encoding['_FillValue']=1.e20 
                ds.lon_segment_002.encoding['_FillValue']=1.e20
                ds.lat_segment_002.encoding['_FillValue']=1.e20
                ds.ny_segment_002.encoding['_FillValue']=-10000
                ds.nz_segment_002.encoding['_FillValue']=-10000
                ds.nx_segment_002.encoding['_FillValue']=-10000
                for v in ds:
                       ds[v].encoding['_FillValue']=1.e20 
                ds = ds.transpose('time','nz_segment_002','ny_segment_002','nx_segment_002','lon_segment_002','lat_segment_002')
                ds.to_netcdf(out_obc_format_dir+'combined_'+new_var_names[var]+'_00'+str(segs[s])+'_'+year+'.nc',unlimited_dims=('time'))
            else:
                nz_zeta =np.linspace(0,1,1)
                dz_zeta = inter_z[:,:,0:1:,:].transpose('ny_segment_002','nz_segment_002','time','nx_segment_002')
                da =  vars()['combined_'+new_var_names[var]+'_00'+str(segs[s])].rename({'nxp':'nx_segment_002'}).expand_dims(dim={'nz_segment_002':nz_zeta,'ny_segment_002':ny_segment_002})
                ds = xr.Dataset(
                        data_vars = dict(
                            zeta_segment_002 = (['ny_segment_002','nz_segment_002','time','nx_segment_002'],da.data),
                            dz_zeta_segment_002 = (['ny_segment_002','nz_segment_002','time','nx_segment_002'],dz_zeta.data),
                            lon_segment_002 = (['lon_segment_002'],lon_segment_002),
                            lat_segment_002 = (['lat_segment_002'],lat_segment_002),
                        ),
                        coords = dict(
                            time = (['time'],da.time.values),
                            nz_segment_002 = (['nz_segment_002'], nz_zeta),
                            ny_segment_002 = (['ny_segment_002'], ny_segment_002),
                            nx_segment_002 = (['nx_segment_002'], nx_segment_002),))
                ds.time.attrs['calendar']='gregorian'
                ds.time.attrs['units']='days since 1999-12-31'
                ds.time.encoding['_FillValue']=1.e20 
                ds.lon_segment_002.encoding['_FillValue']=1.e20
                ds.lat_segment_002.encoding['_FillValue']=1.e20
                ds.ny_segment_002.encoding['_FillValue']=-10000
                ds.nz_segment_002.encoding['_FillValue']=-10000
                ds.nx_segment_002.encoding['_FillValue']=-10000
                for v in ds:
                        ds[v].encoding['_FillValue']=1.e20  
                ds = ds.transpose('time','nz_segment_002','ny_segment_002','nx_segment_002','lon_segment_002','lat_segment_002')
                ds.to_netcdf(out_obc_format_dir+'combined_'+new_var_names[var]+'_00'+str(segs[s])+'_'+year+'.nc',unlimited_dims=('time'))
        elif segs[s] == 4:
            if new_var_names[var] != 'zeta':
                da =  vars()['combined_'+new_var_names[var]+'_00'+str(segs[s])].rename({'depth':'nz_segment_002','nxp':'nx_segment_004'}).expand_dims(dim={'ny_segment_004':ny_segment_004})
                inter_z = inter.rename({'depth':'nz_segment_004'})
                inter_z = inter_z.expand_dims({'ny_segment_004':ny_segment_004,'time':da.time.values,'nx_segment_004':nx_segment_004})
                inter_z = inter_z.transpose('ny_segment_004','time','nz_segment_004','nx_segment_004')
                ds = xr.Dataset(
                    data_vars = dict(
                        {new_var_names[var]+'_segment_004' : (['ny_segment_004','time','nz_segment_004','nx_segment_004'],da.data),
                        'dz_'+new_var_names[var]+'_segment_004': (['ny_segment_004','time','nz_segment_004','nx_segment_004'],inter_z.data),
                        'lon_segment_004' : (['lon_segment_004'],lon_segment_004),
                        'lat_segment_004' : (['lat_segment_004'],lat_segment_004),
                        }),
                    coords = dict(
                        time = (['time'],da.time.values),
                        nz_segment_004 = (['nz_segment_004'], nz_segment_004),
                        ny_segment_004 = (['ny_segment_004'], ny_segment_004),
                        nx_segment_004 = (['nx_segment_004'], nx_segment_004),))
                ds.time.attrs['calendar']='gregorian'
                ds.time.attrs['units']='days since 1999-12-31'
                ds.time.encoding['_FillValue']=1.e20 
                ds.lon_segment_004.encoding['_FillValue']=1.e20
                ds.lat_segment_004.encoding['_FillValue']=1.e20
                ds.ny_segment_004.encoding['_FillValue']=-10000
                ds.nz_segment_004.encoding['_FillValue']=-10000
                ds.nx_segment_004.encoding['_FillValue']=-10000
                for v in ds:
                       ds[v].encoding['_FillValue']=1.e20 
                ds = ds.transpose('time','nz_segment_004','ny_segment_004','nx_segment_004','lon_segment_004','lat_segment_004')
                ds.to_netcdf(out_obc_format_dir+'combined_'+new_var_names[var]+'_00'+str(segs[s])+'_'+year+'.nc',unlimited_dims=('time'))
            else:
                nz_zeta =np.linspace(0,1,1)
                dz_zeta = inter_z[:,:,0:1:,:].transpose('ny_segment_004','nz_segment_004','time','nx_segment_004')
                da =  vars()['combined_'+new_var_names[var]+'_00'+str(segs[s])].rename({'nxp':'nx_segment_004'}).expand_dims(dim={'nz_segment_004':nz_zeta,'ny_segment_004':ny_segment_004})
                ds = xr.Dataset(
                        data_vars = dict(
                            zeta_segment_004 = (['ny_segment_004','nz_segment_004','time','nx_segment_004'],da.data),
                            dz_zeta_segment_004 = (['ny_segment_004','nz_segment_004','time','nx_segment_004'],dz_zeta.data),
                            lon_segment_004 = (['lon_segment_004'],lon_segment_004),
                            lat_segment_004 = (['lat_segment_004'],lat_segment_004),
                        ),
                        coords = dict(
                            time = (['time'],da.time.values),
                            nz_segment_004 = (['nz_segment_004'], nz_zeta),
                            ny_segment_004 = (['ny_segment_004'], ny_segment_004),
                            nx_segment_004 = (['nx_segment_004'], nx_segment_004),))
                ds.time.attrs['calendar']='gregorian'
                ds.time.attrs['units']='days since 1999-12-31'
                ds.time.encoding['_FillValue']=1.e20 
                ds.lon_segment_004.encoding['_FillValue']=1.e20
                ds.lat_segment_004.encoding['_FillValue']=1.e20
                ds.ny_segment_004.encoding['_FillValue']=-10000
                ds.nz_segment_004.encoding['_FillValue']=-10000
                ds.nx_segment_004.encoding['_FillValue']=-10000
                for v in ds:
                        ds[v].encoding['_FillValue']=1.e20  
                ds = ds.transpose('time','nz_segment_004','ny_segment_004','nx_segment_004','lon_segment_004','lat_segment_004')
                ds.to_netcdf(out_obc_format_dir+'combined_'+new_var_names[var]+'_00'+str(segs[s])+'_'+year+'.nc',unlimited_dims=('time'))